# le fat chaton — smol-fat MoE training (HF Hub VM-hopping)

Trains the **smol-fat** profile (~240M total / ~63M-active MoE) and pushes
resumable checkpoints to HuggingFace Hub `mateo0093/le-fat-chaton-ckpt` every
250 steps. If the Colab VM dies (it will, ~90 min idle), just re-run cell 8:
`CHATON_RESUME=1` pulls the latest checkpoint from the Hub and continues from
that exact step — no progress lost (optimizer momentum + step + LR-schedule all
preserved). Proven end-to-end on 2026-07-02 before this notebook was written.

GPU: free T4 16GB. ~1000 steps ≈ 30-60 min. This is the durability + MoE proof
run before committing to the full 10.25B "le fat chaton" on a bigger GPU.

In [ ]:
# 1. Check GPU — want a T4 (16GB) or better. smol-fat MoE fits here.
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 2. Get the code onto the VM.
REPO = 'https://github.com/Mateooo93/le-gros-chaton.git'
%cd /content
!rm -rf le-gros-chaton
!git clone {REPO} le-gros-chaton
%cd /content/le-gros-chaton
!ls -la

In [ ]:
# 3. Install deps — PIN datasets + huggingface_hub (newer versions break wikitext).
!pip install -q tiktoken tokenizers "datasets==2.21.0" "huggingface_hub<0.30"
# IMPORTANT: Runtime -> Restart session after this, then continue from cell 4.
# (Colab keeps the cloned repo across a restart; you don't re-clone.)

After the restart, re-enter the project dir and **set your HF token** so the
notebook can push/pull checkpoints to the Hub. Paste your token in cell 4 (the
notebook never prints it back, and committing the notebook with a fake
placeholder is intentional — never put a real token in a committed file).

In [ ]:
# 4. Re-enter repo (after restart) + set HF token for VM-hopping.
import os
%cd /content/le-gros-chaton

# Paste your HuggingFace WRITE token here (huggingface.co/settings/tokens).
HF_TOKEN = "hf_PASTE_YOUR_WRITE_TOKEN_HERE"
assert HF_TOKEN != "hf_PASTE_YOUR_WRITE_TOKEN_HERE", "paste your real HF write token above"

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['CHATON_HF_REPO'] = 'mateo0093/le-fat-chaton-ckpt'

from huggingface_hub import HfApi
print('whoami:', HfApi().whoami(token=HF_TOKEN)['name'], '(expect mateo0093)')

In [ ]:
# 5. Training overrides — smol-fat MoE on a T4.
import os
os.environ['CHATON_PROFILE']       = 'smol-fat'    # 240M / 63M-active MoE
os.environ['CHATON_MICRO_BATCH']   = '16'
os.environ['CHATON_GRAD_ACCUM']    = '4'           # effective batch = 64
os.environ['CHATON_MAX_ITERS']     = '1000'        # proof run; bump for a real run
os.environ['CHATON_CKPT_INTERVAL'] = '250'         # push to Hub every 250 steps
os.environ['CHATON_RESUME']        = '1'           # PULL latest ckpt from Hub at start
os.environ['CHATON_DATA']          = 'wikitext'    # durability proof on wikitext
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('overrides set: smol-fat MoE, eff batch', 16*4, ', resume from Hub, push every 250')

In [ ]:
# 6. Build the data memmap (wikitext-2, ~700KB download, ~2M tokens — small/fast
#    for the proof). For a real run, set CHATON_CORPUS=wikitext-103 above (~100M tok).
!python -u -c "import data2; print('data ready')" 2>&1 | tail -5

In [ ]:
# 7. Sanity: model builds + start loss ~ ln(vocab)=10.8 (init fix working).
!python -u model.py 2>&1 | tail -6

In [ ]:
# 8. TRAIN — pulls latest ckpt from Hub (resumes if present), trains, pushes
#    every 250 steps. If the VM dies, just re-run THIS cell; it resumes exactly.
!python -u train.py

In [ ]:
# 9. Final save to Google Drive as a backup alongside the Hub checkpoints.
#    (Hub is the primary durable store; Drive is a local convenience copy.)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil, os, glob
    d = '/content/drive/MyDrive/chaton-smol-fat'
    os.makedirs(d, exist_ok=True)
    for f in ['model.pt','checkpoint.pt'] + glob.glob('*.bin'):
        if os.path.exists(f):
            shutil.copy(f, os.path.join(d, f)); print('copied', f)
    print('done ->', d)
except Exception as e:
    print('Drive backup skipped:', e)

In [ ]:
# 10. Sample from the trained MoE.
import torch, os
from model import GPT
from tokenizer import encode, decode
m = GPT().cuda().eval()
p = 'model.pt' if os.path.exists('model.pt') else 'checkpoint.pt'
sd = torch.load(p, map_location='cuda', weights_only=False)
sd = sd['model'] if isinstance(sd,dict) and 'model' in sd else sd
m.load_state_dict(sd)
idx = torch.tensor([encode("The transformer model")], dtype=torch.long, device='cuda')
print(decode(m.generate(idx, max_new_tokens=80, temperature=0.8, top_k=50, top_p=0.9, repetition_penalty=1.2)[0].tolist()))